In [ ]:
%run ./PY_INITIALIZE.ipynb

In [ ]:
from snowflake.ml.registry import Registry
def retrieve_model(model_name):
    #Initialize the model list dataframe
    df_models = []
    filtered_models = []
    reg = Registry(
        session=session, 
        database_name=database_name, 
        schema_name=model_schema
    )
    df_models = reg.show_models()
    if len(df_models)>0:
        filtered_models = df_models[df_models['name'].str.contains(model_name, case=False, na=False)]

    if (len(filtered_models)>0):       
        models_sort = filtered_models.sort_values(by="created_on", ascending=False)
        last2 = models_sort.iloc[0]["name"][-2:]
        model_name = models_sort.iloc[0]["name"]
        model_version = "v"+str(last2)
        model = reg.get_model(model_name).version(model_version)

        
        return model
    else:
        print("Trained model not available")
        return null
    

In [ ]:
    import pandas as pd
    def feature_importance(featureCols):
        # Retrieve the latest version from the registry
        model = retrieve_model("XGBRegressor_clmsPredmodel_0171")
    
        
       # Invoke built-in native training importances attribute of model
        train_importances = model.to_sklearn().feature_importances_
        
        # Format into a readable pandas DataFrame
        train_imp_df = pd.DataFrame({
            'Feature': featureCols,
            'Train_Importance': train_importances
        }).sort_values(by='Train_Importance', ascending=False)

        return train_imp_df
    
    
    

In [ ]:
from sklearn.inspection import permutation_importance 

def testdata_feature_inportance(test_df_pd, features, target):
    
    # 2. Separate your feature columns (X) and target column (y)
    X_test = test_df_pd[feature_names]
    y_test = test_df_pd['TARGET_COLUMN_NAME']
    
    # 3. Pull the underlying scikit-learn model object
    native_model = regressor.to_sklearn()
    
    # 4. Compute permutation importance against the test data
    perm_result = permutation_importance(
        native_model, 
        X_test, 
        y_test, 
        n_repeats=10, 
        random_state=42
    )
    
    # 5. Format into a readable pandas DataFrame
    test_imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Test_Importance_Mean': perm_result.importances_mean,
        'Test_Importance_Std': perm_result.importances_std
    }).sort_values(by='Test_Importance_Mean', ascending=False)
    
    return test_imp_df


In [ ]:
import modin.pandas as pd
import snowflake.snowpark.modin.plugin
def calculate_compare_feature_importances():
    results = session.sql(f"SHOW TABLES LIKE 'TESTING_DATA' in {database_name}.{schema_name}").collect()
    if len(results)>0:
        # file exists
        testData = pd.read_snowflake(f"{database_name}.{schema_name}.TESTING_DATA")
        testData.fillna(0, inplace=True)
        testData_pd = testData.to_pandas()
    features = ['PT_AGE', 'PT_ZIP','ICDCD_NUMCODED','CLM_DIS_RISK_NBR','SUBCD_NBR']
    target = "ClmCostPredictions_2026"
    trainData_featureImp = feature_importance(features)
    print(trainData_featureImp)
    testData_featureImp = testdata_feature_inportance(testData_pd, features, target)
    print(testData_featureImp)   
    
    
    

In [ ]:
calculate_compare_feature_importances()